# Bases de Datos de Grafos y Neo4j para IA

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/graphrag/01-grafos-y-neo4j.ipynb)

Introducción práctica a Neo4j para proyectos de IA: modelado de grafos de conocimiento, Cypher y algoritmos de grafos.

## ¿Qué aprenderás?

- Por qué las bases de datos de grafos son ideales para IA conversacional y RAG
- Modelar entidades y relaciones con Neo4j
- Escribir consultas Cypher para explorar conexiones
- Aplicar algoritmos de grafos (PageRank) para encontrar nodos importantes

## Requisitos

- Python 3.10+
- Neo4j AuraDB (cuenta gratuita) **o** Neo4j Desktop local
- Credenciales: `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`

In [ ]:
# Instalar dependencias
# pip install neo4j pandas matplotlib networkx

import os
import json
from typing import Optional

# Para simulación sin Neo4j real
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

print("Dependencias cargadas correctamente")
print(f"NetworkX version: {nx.__version__}")

## 1. Conexión a Neo4j

Usamos una clase `Neo4jConnector` que encapsula la conexión y las operaciones básicas.
Si no tienes Neo4j disponible, el notebook incluye un modo simulado con NetworkX.

In [ ]:
class Neo4jConnector:
    """
    Conector para Neo4j con soporte para modo simulado (NetworkX).
    
    En producción, reemplaza modo_simulado=False y proporciona credenciales reales.
    """

    def __init__(
        self,
        uri: str = None,
        user: str = None,
        password: str = None,
        modo_simulado: bool = True,
    ):
        self.modo_simulado = modo_simulado
        self.driver = None
        self._grafo_local = nx.DiGraph()  # Grafo en memoria para simulación

        if not modo_simulado:
            try:
                from neo4j import GraphDatabase

                _uri = uri or os.getenv("NEO4J_URI", "bolt://localhost:7687")
                _user = user or os.getenv("NEO4J_USER", "neo4j")
                _password = password or os.getenv("NEO4J_PASSWORD", "password")
                self.driver = GraphDatabase.driver(_uri, auth=(_user, _password))
                self.driver.verify_connectivity()
                print(f"Conectado a Neo4j: {_uri}")
            except Exception as e:
                print(f"No se pudo conectar a Neo4j: {e}")
                print("Cambiando a modo simulado...")
                self.modo_simulado = True
        else:
            print("Modo simulado activado (NetworkX). Sin conexión real a Neo4j.")

    def ejecutar_cypher(self, query: str, params: dict = None) -> list:
        """Ejecuta una consulta Cypher (real o simulada)."""
        if self.modo_simulado:
            return self._ejecutar_simulado(query, params or {})
        with self.driver.session() as session:
            result = session.run(query, params or {})
            return [dict(record) for record in result]

    def _ejecutar_simulado(self, query: str, params: dict) -> list:
        """Simula consultas Cypher básicas sobre el grafo NetworkX."""
        query_upper = query.upper().strip()
        if "CREATE" in query_upper or "MERGE" in query_upper:
            return [{"status": "OK (simulado)"}]
        if "MATCH" in query_upper and "RETURN" in query_upper:
            nodos = list(self._grafo_local.nodes(data=True))
            return [{"nodo": n, "props": d} for n, d in nodos[:10]]
        return []

    def agregar_nodo(self, nombre: str, tipo: str, props: dict = None):
        """Agrega un nodo al grafo simulado."""
        datos = {"tipo": tipo, **(props or {})}
        self._grafo_local.add_node(nombre, **datos)

    def agregar_relacion(self, origen: str, destino: str, tipo: str, props: dict = None):
        """Agrega una relación (arista) al grafo simulado."""
        self._grafo_local.add_edge(origen, destino, tipo=tipo, **(props or {}))

    def cerrar(self):
        if self.driver:
            self.driver.close()


# Instanciar en modo simulado (cambia a False si tienes Neo4j real)
neo4j = Neo4jConnector(modo_simulado=True)
print("Conector listo.")

## 2. Modelar un grafo de empresa

Crearemos un grafo que representa:
- **Empleados** (personas)
- **Proyectos** (iniciativas)
- **Tecnologías** (herramientas y lenguajes)

Con relaciones: `TRABAJA_EN`, `CONOCE`, `USA_TECNOLOGIA`

In [ ]:
# --- Datos de ejemplo ---
empleados = [
    {"nombre": "Ana García", "rol": "ML Engineer", "seniority": "Senior"},
    {"nombre": "Carlos López", "rol": "Backend Developer", "seniority": "Mid"},
    {"nombre": "Elena Martínez", "rol": "Data Scientist", "seniority": "Senior"},
    {"nombre": "David Torres", "rol": "DevOps Engineer", "seniority": "Junior"},
    {"nombre": "Sofía Ruiz", "rol": "Product Manager", "seniority": "Senior"},
]

proyectos = [
    {"nombre": "GraphRAG Engine", "estado": "activo", "prioridad": "alta"},
    {"nombre": "Chatbot Legal", "estado": "activo", "prioridad": "media"},
    {"nombre": "Pipeline MLOps", "estado": "completado", "prioridad": "alta"},
]

tecnologias = [
    {"nombre": "Python", "categoria": "lenguaje"},
    {"nombre": "Neo4j", "categoria": "base_datos"},
    {"nombre": "PyTorch", "categoria": "ml_framework"},
    {"nombre": "Kubernetes", "categoria": "infraestructura"},
    {"nombre": "Claude API", "categoria": "llm"},
    {"nombre": "FastAPI", "categoria": "framework_web"},
]

# Agregar nodos
for emp in empleados:
    neo4j.agregar_nodo(emp["nombre"], "Empleado", emp)

for proy in proyectos:
    neo4j.agregar_nodo(proy["nombre"], "Proyecto", proy)

for tec in tecnologias:
    neo4j.agregar_nodo(tec["nombre"], "Tecnologia", tec)

# Relaciones empleado -> proyecto
relaciones_trabajo = [
    ("Ana García", "GraphRAG Engine", "TRABAJA_EN", {"horas_semana": 30}),
    ("Carlos López", "GraphRAG Engine", "TRABAJA_EN", {"horas_semana": 20}),
    ("Elena Martínez", "Chatbot Legal", "TRABAJA_EN", {"horas_semana": 25}),
    ("Ana García", "Chatbot Legal", "TRABAJA_EN", {"horas_semana": 10}),
    ("David Torres", "Pipeline MLOps", "TRABAJA_EN", {"horas_semana": 40}),
    ("Sofía Ruiz", "GraphRAG Engine", "TRABAJA_EN", {"horas_semana": 5}),
]

# Relaciones empleado -> tecnología
relaciones_tech = [
    ("Ana García", "Python", "CONOCE", {"nivel": "experto"}),
    ("Ana García", "PyTorch", "CONOCE", {"nivel": "avanzado"}),
    ("Ana García", "Claude API", "CONOCE", {"nivel": "avanzado"}),
    ("Carlos López", "Python", "CONOCE", {"nivel": "avanzado"}),
    ("Carlos López", "FastAPI", "CONOCE", {"nivel": "experto"}),
    ("Carlos López", "Neo4j", "CONOCE", {"nivel": "intermedio"}),
    ("Elena Martínez", "Python", "CONOCE", {"nivel": "experto"}),
    ("Elena Martínez", "PyTorch", "CONOCE", {"nivel": "experto"}),
    ("David Torres", "Kubernetes", "CONOCE", {"nivel": "experto"}),
    ("David Torres", "Python", "CONOCE", {"nivel": "intermedio"}),
]

# Relaciones proyecto -> tecnología
relaciones_proy_tech = [
    ("GraphRAG Engine", "Python", "USA_TECNOLOGIA", {}),
    ("GraphRAG Engine", "Neo4j", "USA_TECNOLOGIA", {}),
    ("GraphRAG Engine", "Claude API", "USA_TECNOLOGIA", {}),
    ("Chatbot Legal", "Python", "USA_TECNOLOGIA", {}),
    ("Chatbot Legal", "Claude API", "USA_TECNOLOGIA", {}),
    ("Chatbot Legal", "FastAPI", "USA_TECNOLOGIA", {}),
    ("Pipeline MLOps", "Kubernetes", "USA_TECNOLOGIA", {}),
    ("Pipeline MLOps", "Python", "USA_TECNOLOGIA", {}),
]

for o, d, t, p in relaciones_trabajo + relaciones_tech + relaciones_proy_tech:
    neo4j.agregar_relacion(o, d, t, p)

print(f"Grafo creado: {neo4j._grafo_local.number_of_nodes()} nodos, "
      f"{neo4j._grafo_local.number_of_edges()} relaciones")

## 3. Visualización del grafo

Representamos el grafo con colores por tipo de nodo: empleados (azul), proyectos (verde), tecnologías (naranja).

In [ ]:
G = neo4j._grafo_local

# Colores por tipo de nodo
color_map = {
    "Empleado": "#4A90D9",
    "Proyecto": "#2ECC71",
    "Tecnologia": "#E67E22",
}
node_colors = [color_map.get(G.nodes[n].get("tipo", ""), "#999") for n in G.nodes]

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42, k=2.5)

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=800, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=8, font_color="white", font_weight="bold")
nx.draw_networkx_edges(
    G, pos,
    edge_color="#AAAAAA",
    arrows=True,
    arrowsize=15,
    width=1.5,
    connectionstyle="arc3,rad=0.1",
)

# Leyenda
leyenda = [mpatches.Patch(color=v, label=k) for k, v in color_map.items()]
plt.legend(handles=leyenda, loc="upper left", fontsize=10)
plt.title("Grafo de conocimiento de la empresa", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Consultas Cypher equivalentes

Así se escribirían las consultas en Neo4j real. En modo simulado las ejecutamos sobre el grafo NetworkX.

In [ ]:
# ----- Consulta 1: todos los empleados -----
cypher_empleados = """
MATCH (e:Empleado)
RETURN e.nombre AS nombre, e.rol AS rol, e.seniority AS nivel
ORDER BY e.seniority
"""
print("=== Consulta Cypher (simulada) ===")
print(cypher_empleados)

# Simulación manual con NetworkX
empleados_resultado = [
    {"nombre": n, "rol": d.get("rol"), "nivel": d.get("seniority")}
    for n, d in G.nodes(data=True)
    if d.get("tipo") == "Empleado"
]
df_empleados = pd.DataFrame(empleados_resultado)
print(df_empleados.to_string(index=False))

print()

# ----- Consulta 2: caminos (empleado -> proyecto -> tecnología) -----
cypher_caminos = """
MATCH (e:Empleado)-[:TRABAJA_EN]->(p:Proyecto)-[:USA_TECNOLOGIA]->(t:Tecnologia)
RETURN e.nombre AS empleado, p.nombre AS proyecto, t.nombre AS tecnologia
ORDER BY empleado
"""
print("=== Caminos empleado -> proyecto -> tecnología ===")
print(cypher_caminos)

# Simulación: buscar caminos de longitud 2
caminos = []
for empleado in [n for n, d in G.nodes(data=True) if d.get("tipo") == "Empleado"]:
    for proyecto in G.successors(empleado):
        if G.nodes[proyecto].get("tipo") == "Proyecto":
            for tecnologia in G.successors(proyecto):
                if G.nodes[tecnologia].get("tipo") == "Tecnologia":
                    caminos.append(
                        {"empleado": empleado, "proyecto": proyecto, "tecnologia": tecnologia}
                    )

df_caminos = pd.DataFrame(caminos)
print(df_caminos.to_string(index=False))

## 5. Algoritmo PageRank sobre el grafo

PageRank nos dice qué nodos son más "importantes" (más referenciados). Aplicado a grafos de conocimiento, identifica las tecnologías más críticas o las personas más conectadas.

In [ ]:
# Calcular PageRank
pagerank_scores = nx.pagerank(G, alpha=0.85, max_iter=200)

# Crear DataFrame con scores y tipo
df_pr = pd.DataFrame([
    {
        "nodo": nodo,
        "tipo": G.nodes[nodo].get("tipo", "?"),
        "pagerank": round(score, 4),
    }
    for nodo, score in pagerank_scores.items()
]).sort_values("pagerank", ascending=False)

print("=== Top 10 nodos por PageRank ===")
print(df_pr.head(10).to_string(index=False))

# Visualización: tamaño del nodo proporcional al PageRank
plt.figure(figsize=(14, 9))

node_sizes = [pagerank_scores[n] * 15000 + 300 for n in G.nodes]
node_colors = [color_map.get(G.nodes[n].get("tipo", ""), "#999") for n in G.nodes]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.85)
nx.draw_networkx_labels(G, pos, font_size=7, font_color="white", font_weight="bold")
nx.draw_networkx_edges(
    G, pos, edge_color="#CCCCCC", arrows=True, arrowsize=12,
    width=1.2, connectionstyle="arc3,rad=0.1"
)

leyenda = [mpatches.Patch(color=v, label=k) for k, v in color_map.items()]
plt.legend(handles=leyenda, loc="upper left", fontsize=10)
plt.title("PageRank — tamaño de nodo proporcional a importancia", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

# Insight clave
top_nodo = df_pr.iloc[0]
print(f"\nNodo mas importante: '{top_nodo['nodo']}' "
      f"(tipo: {top_nodo['tipo']}, PageRank: {top_nodo['pagerank']})")

## Resumen y próximos pasos

### Lo que hemos visto

| Concepto | Descripción |
|----------|-------------|
| **Neo4jConnector** | Clase reutilizable con modo real y simulado |
| **Modelado** | Nodos (Empleado, Proyecto, Tecnología) + relaciones tipadas |
| **Cypher** | Lenguaje declarativo para consultar grafos |
| **PageRank** | Algoritmo para identificar nodos importantes |

### ¿Por qué usar grafos para RAG?

- Los vectores buscan **semejanza semántica**, los grafos buscan **conexiones estructuradas**
- Las preguntas como "¿quién trabaja con quién en qué proyecto?" son naturales en grafos
- GraphRAG combina ambos mundos para respuestas más ricas y contextualizadas

### Próximos pasos

- **Notebook 02**: Pipeline completo de Microsoft GraphRAG
- **Notebook 03**: Extracción automática de grafos con Claude
- **Notebook 04**: GraphRAG en producción y RAG híbrido

> Para usar Neo4j real, regístrate en [Neo4j AuraDB](https://neo4j.com/cloud/platform/aura-graph-database/) (tier gratuito disponible) y actualiza las variables de entorno.